# 📊 LexGLUE/LEDGAR: Exploratory Data Analysis & Class Imbalance Audit
### Domain-Specific Legal Contract Clause Classification

This notebook explores the **LexGLUE LEDGAR** benchmark subset, which evaluates classification across **100 legal contract provision categories**.

#### Objectives:
1. Inspect split sizes and data schema (`text`, `label`).
2. Quantify class imbalance (Gini coefficient, frequency ratios, top-10 vs. tail-10 classes).
3. Analyze sequence length distribution to justify `max_seq_length=512`.
4. Compute and verify balanced inverse-frequency class weights for our custom `WeightedSFTTrainer`.
5. Examine sample provisions and label mappings.

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from datasets import load_dataset
from sklearn.utils.class_weight import compute_class_weight

# Clean plot styling
plt.style.use(
    "seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default"
)
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11
print("Libraries loaded successfully.")

## 1. Load the LEDGAR Dataset
LEDGAR is hosted under the `coastalcph/lex_glue` repository on Hugging Face.

In [ ]:
raw_dataset = load_dataset("coastalcph/lex_glue", "ledgar")
print(raw_dataset)

label_names = raw_dataset["train"].features["label"].names
print(f"\nTotal provision classes: {len(label_names)}")
print(f"Sample class labels (first 10): {label_names[:10]}")

## 2. Class Distribution & Imbalance Audit
Let's measure how skewed the 100 provision classes are in the 60,000-sample training set.

In [ ]:
train_labels = raw_dataset["train"]["label"]
counter = Counter(train_labels)
num_classes = len(label_names)

freq_array = np.array([counter[i] for i in range(num_classes)])
df_classes = (
    pd.DataFrame(
        {
            "class_id": range(num_classes),
            "class_name": label_names,
            "count": freq_array,
            "percentage": (freq_array / len(train_labels)) * 100,
        }
    )
    .sort_values(by="count", ascending=False)
    .reset_index(drop=True)
)


def compute_gini(frequencies):
    sorted_freqs = np.sort(frequencies)
    n = len(frequencies)
    index = np.arange(1, n + 1)
    return (2 * np.sum(index * sorted_freqs)) / (n * np.sum(sorted_freqs)) - (n + 1) / n


gini = compute_gini(freq_array)
imbalance_ratio = df_classes["count"].max() / max(1, df_classes["count"].min())

print(f"Training set samples: {len(train_labels):,}")
print(
    f"Most frequent class: '{df_classes.iloc[0]['class_name']}' with {df_classes.iloc[0]['count']:,} samples ({df_classes.iloc[0]['percentage']:.2f}%)"
)
print(
    f"Least frequent class: '{df_classes.iloc[-1]['class_name']}' with {df_classes.iloc[-1]['count']:,} samples ({df_classes.iloc[-1]['percentage']:.2f}%)"
)
print(f"Imbalance Ratio (Max/Min): {imbalance_ratio:.2f}x")
print(f"Gini Coefficient: {gini:.4f} (High inequality!)")

In [ ]:
# Visualize Top 15 Majority and Bottom 15 Tail Classes
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

sns.barplot(data=df_classes.head(15), y="class_name", x="count", ax=axes[0], palette="Blues_r")
axes[0].set_title("Top 15 Majority Legal Categories", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Number of Provisions")
axes[0].set_ylabel("Provision Category")

sns.barplot(data=df_classes.tail(15), y="class_name", x="count", ax=axes[1], palette="Reds_r")
axes[1].set_title("Bottom 15 Minority (Tail) Legal Categories", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Number of Provisions")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

## 3. Sequence Length Distribution
Phi-3-mini has a native 4,000-token context window, but using `max_seq_length=512` for sequence classification saves massive VRAM during training. Let's verify that 512 covers almost all contract clauses.

In [ ]:
# Sample 10,000 provisions to compute word lengths
sample_texts = raw_dataset["train"]["text"][:10000]
word_lengths = np.array([len(t.split()) for t in sample_texts])

p50, p90, p95, p99 = np.percentile(word_lengths, [50, 90, 95, 99])
print("Word count percentiles:")
print(f"  50th (Median): {p50:.1f} words")
print(f"  90th:          {p90:.1f} words")
print(f"  95th:          {p95:.1f} words")
print(f"  99th:          {p99:.1f} words")

plt.figure(figsize=(10, 4))
sns.histplot(word_lengths, bins=60, kde=True, color="teal")
plt.axvline(x=400, color="red", linestyle="--", label="~512 Token Equivalent (~400 words)")
plt.title("Provision Word Length Distribution", fontsize=13, fontweight="bold")
plt.xlabel("Word Count")
plt.ylabel("Clause Count")
plt.legend()
plt.show()

## 4. Balanced Inverse-Frequency Class Weights
We calculate exact balanced class weights:
$$w_c = \frac{N}{C \cdot N_c}$$
And normalize so the average weight is 1.0. These weights will be passed directly to our `WeightedSFTTrainer`.

In [ ]:
classes = np.arange(num_classes)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_labels,
)

weights_normalized = weights / np.mean(weights)

df_weights = pd.DataFrame(
    {
        "class_id": range(num_classes),
        "class_name": label_names,
        "raw_weight": weights,
        "normalized_weight": weights_normalized,
    }
).sort_values(by="normalized_weight", ascending=False)

print("Highest Weighted Classes (Minority Boost):")
print(df_weights.head(5)[["class_name", "normalized_weight"]])

print("\nLowest Weighted Classes (Majority Down-weight):")
print(df_weights.tail(5)[["class_name", "normalized_weight"]])

## 5. Sample Provision Inspection & Phi-3 Prompt Formatting Preview
Let's see how raw provisions map to our Phi-3 instruct format.

In [ ]:
sample = raw_dataset["train"][0]
sample_text = sample["text"]
sample_label_id = sample["label"]
sample_label_name = label_names[sample_label_id]

phi3_prompt = (
    "<|user|>\n"
    "You are an expert legal AI assistant. Classify the following contract provision into exactly one of the 100 standard LEDGAR provision categories. Respond with only the exact category name.\n\n"
    f'Contract Provision:\n"{sample_text}"\n<|end|>\n'
    "<|assistant|>\n"
    f"{sample_label_name}<|end|>"
)

print("=== FORMATTED PHI-3 INSTRUCTION PROMPT PREVIEW ===")
print(phi3_prompt)